# Tidal Dataset Generation

This notebook generates the final dataset of highs and lows in tidal patterns from NOAA and USGS between 2025 and 2026. The following six stations were identified as NYC stations with tidal gauges:
- NOAA: The Battery, Kings Point
- USGS: Great Kills Harbor at Great Kills NY, East Rockaway Inlet at Atlantic Beach NY, Jamaica bay at Inwood NY, Rockaway Inlet near Floyd Bennett Field NY

Data was impored directly from NOAA and USGS libraries. The final dataset is used in 2_tidal_analysis.ipynb.

In [23]:
# import libraries and read in API keys

import os
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

NOAA_API  = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"
NOAA_META = "https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{}.json"
YEARS = [2025, 2026]

# Set True to re-fetch from NOAA/USGS even if a cached file already exists on disk
FORCE_REFRESH = False


In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

STATION_IDS = ["8518750", "8516945"]  # The Battery, Kings Point


def _noaa_session():
    # Session with automatic retries on rate-limit (429) and server-side (5xx) errors,
    # using exponential backoff so repeated failures don't hammer the API.
    s = requests.Session()
    retry = Retry(total=3, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retry))
    return s


def station_meta(station_id):
    # Pull station name and lat/lng from NOAA's metadata endpoint
    r = _noaa_session().get(NOAA_META.format(station_id), timeout=30)
    r.raise_for_status()
    s = r.json()["stations"][0]
    return {"name": s["name"], "lat": float(s["lat"]), "lng": float(s["lng"])}


def fetch_predictions(station_id, year):
    """Fetch high/low predictions one month at a time to avoid API timeouts"""
    session = _noaa_session()
    months = []
    for month in range(1, 13):
        begin = pd.Timestamp(year, month, 1)
        end   = begin + pd.offsets.MonthEnd(0)
        r = session.get(NOAA_API, params={
            "product":    "predictions",
            "begin_date": begin.strftime("%Y%m%d"),
            "end_date":   end.strftime("%Y%m%d"),
            "datum":      "MLLW",
            "station":    station_id,
            "time_zone":  "lst_ldt",
            "interval":   "hilo",
            "units":      "english",
            "format":     "json",
        }, timeout=30)
        r.raise_for_status()
        months.append(pd.DataFrame(r.json()["predictions"]))
     # Combine all 12 months into one DataFrame for the year
    return pd.concat(months, ignore_index=True)


NOAA_CACHE = r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis\tide_predictions.geojson"

if not FORCE_REFRESH and os.path.exists(NOAA_CACHE):
    tides_gdf = gpd.read_file(NOAA_CACHE)
    tides_gdf["datetime"] = pd.to_datetime(tides_gdf["datetime"])
    # GeoJSON round-tripping drops "time" silently: OGR's type-sniffer misreads "07:45 AM"-style
    # strings as its own unsupported Time field type. Regenerate all three derived display
    # columns from "datetime" rather than relying on what survived the round-trip.
    tides_gdf["date"] = tides_gdf["datetime"].dt.strftime("%Y/%m/%d")
    tides_gdf["day"]  = tides_gdf["datetime"].dt.strftime("%a")
    tides_gdf["time"] = tides_gdf["datetime"].dt.strftime("%I:%M %p")
    print(f"Loaded cached NOAA predictions from {NOAA_CACHE}")
else:
    frames = []
    for sid in STATION_IDS:
        meta = station_meta(sid)
        # Fetch predictions for every configured year and stack them into one DataFrame per station
        df   = pd.concat([fetch_predictions(sid, yr) for yr in YEARS], ignore_index=True)

        df["datetime"]     = pd.to_datetime(df["t"], format="%Y-%m-%d %H:%M")
        df["pred_in_ft"]   = df["v"].astype(float)
        df["pred_in_cm"]   = (df["pred_in_ft"] * 30.48).round().astype(int)
        df["date"]         = df["datetime"].dt.strftime("%Y/%m/%d")
        df["day"]          = df["datetime"].dt.strftime("%a")
        df["time"]         = df["datetime"].dt.strftime("%I:%M %p")
        df["highlow"]      = df["type"]
        df["station_id"]   = sid
        df["station_name"] = meta["name"]
        df["geometry"]     = Point(meta["lng"], meta["lat"])

        # Keep only the standardized columns before appending to the combined list
        frames.append(df[["datetime", "date", "day", "time",
                           "pred_in_ft", "pred_in_cm", "highlow",
                           "station_id", "station_name", "geometry"]])

    # Combine both stations into a single GeoDataFrame
    tides_gdf = gpd.GeoDataFrame(
        pd.concat(frames, ignore_index=True),
        geometry="geometry",
        crs="EPSG:4326",
    )


In [ ]:
for sid, group in tides_gdf.groupby("station_id", sort=False):
    print(f"\n── {group['station_name'].iloc[0]}  ({sid})  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["date", "day", "time", "pred_in_ft", "pred_in_cm", "highlow"]]
              .head(8)
              .reset_index(drop=True))

print(f"\nTotal records: {len(tides_gdf)}  |  CRS: {tides_gdf.crs}")

tides_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis\tide_predictions.geojson",
    driver="GeoJSON",
)
print("Saved → tide_predictions.geojson")

We now have NOAA's datasets for The Battery and King's Point, which will be combined with USGS data later in this file.

## USGS Tidal Observations (parameter 62620 — water surface elevation above NAVD 1988)

We now import USGS data from the 4 stations listed in the introduction of this file. For this analysis, the North American Vertical Datum of 1988 was used as the metric for all four tidal gauges due to data availability and the fact that this is the most updated metric used by USGS in calculating tidal wave predictions.

In [ ]:
import dataretrieval.nwis as nwis

USGS_SITES = {
    "01311875": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311850": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01311145": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet NAVD88
    "01376562": "62620",  # Estuary or ocean water surface elevation above NAVD 1988, feet
}

START_DT = "2025-01-01"
# Always fetch through today's date on a forced refresh, so re-running with FORCE_REFRESH=True picks up new data
END_DT = pd.Timestamp.today().strftime("%Y-%m-%d")


def fetch_usgs_meta(site_no):
    # "site" service returns station metadata only, not any time-series values
    df = nwis.get_record(sites=site_no, service="site")
    row = df.iloc[0]
    return {
        "name": row["station_nm"],
        "lat":  float(row["dec_lat_va"]),
        "lon":  float(row["dec_long_va"]),
    }


def fetch_usgs_wl(site_no, param, start_dt, end_dt):
    """Fetch USGS instantaneous values one month at a time to avoid gateway timeouts."""
    start, end = pd.Timestamp(start_dt), pd.Timestamp(end_dt)
    chunks = []
    cursor = start
    while cursor <= end:
         # Cap each chunk at one month or the overall end date, whichever comes first
        chunk_end = min(cursor + pd.DateOffset(months=1) - pd.Timedelta(days=1), end)
        df, _ = nwis.get_iv(
            sites=site_no, parameterCd=param,
            startDT=cursor.strftime("%Y-%m-%d"),
            endDT=chunk_end.strftime("%Y-%m-%d"),
        )
        if not df.empty:
            # Column names embed the parameter code and can have suffixes (e.g. "62620_00000"),
            # so match dynamically rather than hardcoding a name; exclude "_cd" quality-code columns
            value_col = next(
                c for c in df.columns
                if (c == param or c.startswith(param + "_")) and not c.endswith("_cd")
            )
            df = df[[value_col]].rename(columns={value_col: "value"})
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            # nwis.get_iv returns the timestamp as the index; name it before resetting so it becomes a column
            df.index.name = "datetime"
            chunks.append(df.reset_index())
        cursor += pd.DateOffset(months=1)
    return pd.concat(chunks, ignore_index=True)


USGS_CACHE = r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis\usgs_water_levels.geojson"

if not FORCE_REFRESH and os.path.exists(USGS_CACHE):
    # Skip the live USGS fetch and reuse the on-disk copy. Set FORCE_REFRESH = True above to
    # pull fresh readings (needed since END_DT tracks "today" — the cache won't include any
    # data newer than whenever it was last fetched).
    usgs_gdf = gpd.read_file(USGS_CACHE)
    usgs_gdf["datetime"] = pd.to_datetime(usgs_gdf["datetime"])
    print(f"Loaded cached USGS water levels from {USGS_CACHE}")
else:
    frames = []
    # One fetch per site; each site's rows are collected into `frames`, then concatenated below
    for site_no, param_cd in USGS_SITES.items():
        meta = fetch_usgs_meta(site_no)
        df   = fetch_usgs_wl(site_no, param_cd, START_DT, END_DT)
        df["site_no"]      = site_no
        df["param_cd"]     = param_cd
        df["station_name"] = meta["name"]
         # shapely Point expects (x, y) = (lon, lat)
        df["geometry"]     = Point(meta["lon"], meta["lat"])
        frames.append(df)

    usgs_gdf = gpd.GeoDataFrame(
        pd.concat(frames, ignore_index=True),
        geometry="geometry",
        crs="EPSG:4326",
    )


In [ ]:
for site, group in usgs_gdf.groupby("site_no", sort=False):
    print()
    print(f"── {group['station_name'].iloc[0]}  ({site})  param {group['param_cd'].iloc[0]}  "
          f"{group['geometry'].iloc[0].y:.4f}°N, {group['geometry'].iloc[0].x:.4f}°W ──")
    display(group[["datetime", "value"]].head(8).reset_index(drop=True))

print()
print(f"Total records: {len(usgs_gdf)}  |  CRS: {usgs_gdf.crs}")

usgs_gdf.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis\usgs_water_levels.geojson",
    driver="GeoJSON",
)
print("Saved → usgs_water_levels.geojson")

## Standardized Unified Schema

Both sources are aligned to a common schema:

| column | type | notes |
|---|---|---|
| `datetime` | datetime (UTC, tz-aware) | NOAA localized from US/Eastern; USGS converted to UTC |
| `water_level_ft` | float | **NAVD88 datum** for both sources |
| `tidal_phase` | string | `H` / `L` (NOAA); `rising` / `falling` (USGS) |
| `surge_ft` | float | `0.0` for NOAA predicted; `NaN` for USGS (requires paired predicted) |
| `source` | string | `"noaa_predicted"` or `"usgs_observed"` |
| `station_id` | string | NOAA station ID or USGS site number |
| `station_name` | string | — |
| `geometry` | Point (EPSG:4326) | — |

**Datum conversion (NOAA only):** NOAA predictions are in feet above MLLW. NOAA's own datums API gives the height of each datum above the station datum (STND); the difference `MLLW_stnd − NAVD_stnd` equals how far MLLW is above NAVD88. Subtracting that offset converts every NOAA reading to feet above NAVD88.

In [28]:
def fetch_noaa_datums(station_id):
    r = requests.get(
        f"https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/{station_id}/datums.json",
        params={"units": "english"},
    )
    r.raise_for_status()
    return {d["name"]: float(d["value"]) for d in r.json()["datums"]}


# How far MLLW sits above NAVD88 at each station (feet).
# Negative at NYC stations — MLLW is below NAVD88.
# Adding this to a MLLW-referenced value gives a NAVD88-referenced value.
MLLW_ABOVE_NAVD = {}
for sid in STATION_IDS:
    d = fetch_noaa_datums(sid)
    MLLW_ABOVE_NAVD[sid] = d["MLLW"] - d["NAVD88"]
    print(f"{sid}: MLLW is {MLLW_ABOVE_NAVD[sid]:+.3f} ft above NAVD88")

8518750: MLLW is -2.770 ft above NAVD88
8516945: MLLW is -4.210 ft above NAVD88


### Datum conversion: MLLW → NAVD88

**What the NOAA datums API returns**

`fetch_noaa_datums()` returns every datum for a station as a height **above STND** — the physical benchmark bolt at the gauge. STND is just a local reference pin; it has no global meaning. Both `d["MLLW"]` and `d["NAVD88"]` are measured up from the same STND, so their difference is a meaningful physical offset between the two datums.

**Physical ordering at NYC stations**

NAVD88 approximates mean sea level. MLLW (Mean Lower Low Water) is the average of the lowest daily tides — well below mean sea level. So at every NYC gauge:

```
↑  MHW    (~+2 ft relative to NAVD88)
↑  NAVD88  ≈ mean sea level          d["NAVD88"] > d["MLLW"]
↑  MLLW   (~-2.5 ft relative to NAVD88)
```

Therefore `MLLW_ABOVE_NAVD = d["MLLW"] − d["NAVD88"]` is a **negative number** — it quantifies how far below NAVD88 the MLLW zero mark sits.

**Converting a MLLW-referenced prediction to NAVD88**

NOAA tide predictions (`pred_in_ft`) are heights above the MLLW zero mark. To convert, find the water surface's absolute elevation above STND, then re-reference it to NAVD88:

```
absolute elevation above STND  =  d["MLLW"] + pred_in_ft
elevation above NAVD88         = (d["MLLW"] + pred_in_ft) − d["NAVD88"]
                               =  pred_in_ft + (d["MLLW"] − d["NAVD88"])
                               =  pred_in_ft + MLLW_ABOVE_NAVD
```

Because `MLLW_ABOVE_NAVD` is negative, this **subtracts** the gap between NAVD88 and MLLW — which is exactly what you want. A low-tide reading of 0.3 ft above MLLW becomes `0.3 + (−2.5) = −2.2 ft` relative to NAVD88, correctly placing low water below mean sea level.

**Why `− MLLW_ABOVE_NAVD` is wrong**

Subtracting a negative number adds its magnitude: `pred − (−2.5) = pred + 2.5`. That pushes every reading *further above* MLLW rather than re-referencing it downward to NAVD88 — a physically impossible result for typical tide levels.

In [29]:
import numpy as np

# ── NOAA: MLLW → NAVD88, datetimes localized US/Eastern → UTC ──
noaa_rows = []
for sid, group in tides_gdf.groupby("station_id", sort=False):
    g = group.copy()
    g["water_level_ft"] = g["pred_in_ft"] + MLLW_ABOVE_NAVD[sid]
    # ambiguous="NaT" drops the repeated fall-back hour; nonexistent="shift_forward" skips the missing spring-forward hour
    g["datetime"] = (
        g["datetime"]
        .dt.tz_localize("US/Eastern", ambiguous="NaT", nonexistent="shift_forward")
        .dt.tz_convert("UTC")
    )
    g = g.dropna(subset=["datetime"])
    # Identity map keeps NOAA's H/L labels consistent with the unified schema's tidal_phase values
    g["tidal_phase"] = g["highlow"].map({"H": "H", "L": "L"})
    # Predicted tide has no storm surge component by definition
    g["surge_ft"]    = 0.0
    g["source"]      = "noaa_predicted"
    g["station_id"]  = sid
    noaa_rows.append(g[["datetime", "water_level_ft", "tidal_phase",
                         "surge_ft", "source", "station_id", "station_name", "geometry"]])

noaa_std = gpd.GeoDataFrame(
    pd.concat(noaa_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)


### USGS high/low tide extraction: harmonic analysis (UTide) instead of raw peak-picking

NOAA already publishes clean, pre-computed high/low tide *events* (see the schema table above), so its `tidal_phase` values need no further processing. USGS is different: it only provides a continuous, noisy, *observed* water-level series (a reading every 6–15 minutes) with no separate "predicted tide" to compare against.

**The problem with the old approach.** It inferred `H`/`L` directly from the raw series — any `rising` reading immediately followed by `falling` got labeled a high tide, and vice versa for lows. That fires on wind chop, boat wake, and sensor noise just as readily as on a real tidal peak, since the raw signal isn't smooth. (In practice this produced ~3.6x more "high tide" events than physically expected, including some at physically-impossible low-tide elevations.)

**The fix: fit the known astronomical tidal constituents (M2, S2, N2, K1, O1, …) to each site's observed record with [UTide](https://github.com/wesleybowman/UTide), then find extrema on the *reconstructed*, noise-free curve instead of the raw one.**

Steps, per USGS site:
1. **Fit (`utide.solve`)** — regresses the observed water level onto a fixed set of astronomically-known frequencies. Amplitude and phase are solved from this site's own data (they depend on local bathymetry), but the frequencies themselves are fixed constants, not fitted. Anything that doesn't oscillate at one of those frequencies — sensor noise, wind chop, storm surge — is excluded automatically, not via a manually-tuned threshold.
2. **Reconstruct (`utide.reconstruct`)** — evaluates the fitted constituents back at the original timestamps, producing a smooth, astronomical-only curve.
3. **Surge, as a side effect** — `surge_ft = observed − reconstructed` becomes computable for USGS for the first time (it was `NaN` in the unified schema before, since there was no true tidal reference to diff against).
4. **Find extrema on the smooth curve** — `scipy.signal.find_peaks` is now safe to use directly on the reconstructed curve (and its negation, for lows), because the noise that used to create spurious extra peaks has already been fitted out rather than left in.
5. **Collapse to extrema-only rows** — matches NOAA's shape (one row per real tide event, not one row per 6-minute reading). The in-between `rising`/`falling` readings are dropped; nothing downstream in `2_tidal_analysis.ipynb` used them — it only ever filters on `tidal_phase == 'H'`.

In [30]:
import numpy as np
from scipy.signal import find_peaks
import utide

usgs_rows = []
for site, group in usgs_gdf.groupby("site_no", sort=False):
    g = group.copy().sort_values("datetime").reset_index(drop=True)
    g["water_level_ft"] = g["value"]
    # A handful of readings can come back non-numeric (coerced to NaN during the USGS fetch);
    # utide.solve() can't fit through gaps like that, so drop them before fitting.
    g = g.dropna(subset=["water_level_ft"]).reset_index(drop=True)

    dt = pd.to_datetime(g["datetime"])
    # USGS timestamps may already be tz-aware (UTC) or naive depending on the query; handle both
    g["datetime"] = dt.dt.tz_convert("UTC") if dt.dt.tz is not None else dt.dt.tz_localize("UTC")

    # UTide needs (1) tz-naive datetimes and (2) the station's latitude, for the ~18.6-year lunar
    # nodal correction. Pass real datetimes straight through — do NOT pre-convert to a numeric
    # datenum. utide.solve()'s default (no explicit `epoch=`) re-interprets a raw float array as
    # milliseconds since 1970, not days, which silently collapses the whole record to a span of
    # seconds and makes it fit zero usable constituents with no error or warning.
    # Latitude isn't its own column here, but it's already embedded in the Point geometry set
    # during the USGS fetch (Point(lon, lat), so .y is latitude — same trick the display cell
    # above uses).
    site_lat = g["geometry"].iloc[0].y
    t = g["datetime"].dt.tz_localize(None)

    # Fit ONLY the known astronomical tidal frequencies (M2, S2, N2, K1, O1, ...) to this site's
    # observed record via least squares. Anything that doesn't oscillate at one of those fixed
    # frequencies — sensor noise, wind chop, storm surge — falls out of the fit automatically.
    coef = utide.solve(
        t,
        g["water_level_ft"].to_numpy(),
        lat=site_lat,
        method="ols",
        conf_int="none",
        verbose=False,
    )

    # Reconstruct a smooth, noise-free water-level curve from the fitted constituents, evaluated
    # at the same timestamps as the raw observations.
    recon = utide.reconstruct(t, coef, verbose=False)
    recon_ft = recon.h

    # Surge is whatever the astronomical fit didn't explain: observed minus reconstructed.
    # NOAA's rows are pure predictions and are defined to have zero surge (see the NOAA section
    # above); USGS rows get a real, non-zero value here for the first time.
    g["surge_ft"] = g["water_level_ft"] - recon_ft

    # Find local maxima/minima of the RECONSTRUCTED curve, not the raw one — this is the actual
    # fix for the old bug, since the reconstructed curve has no wind chop/sensor noise left to
    # create spurious extra peaks.
    high_idx, _ = find_peaks(recon_ft)
    low_idx, _  = find_peaks(-recon_ft)

    extrema_idx   = np.concatenate([high_idx, low_idx])
    extrema_phase = np.array(["H"] * len(high_idx) + ["L"] * len(low_idx))

    # Keep only the extrema rows — mirrors NOAA's shape, which only ever has discrete high/low
    # rows to begin with (see the schema table above). The in-between rising/falling readings
    # are dropped rather than carried through unused.
    g_extrema = g.iloc[extrema_idx].copy()
    g_extrema["tidal_phase"] = extrema_phase
    g_extrema = g_extrema.sort_values("datetime").reset_index(drop=True)

    g_extrema["source"]     = "usgs_observed"
    g_extrema["station_id"] = site
    usgs_rows.append(g_extrema[["datetime", "water_level_ft", "tidal_phase",
                                 "surge_ft", "source", "station_id", "station_name", "geometry"]])

usgs_std = gpd.GeoDataFrame(
    pd.concat(usgs_rows, ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)


In [ ]:
# ── Combine ──
tidal_unified = gpd.GeoDataFrame(
    pd.concat([noaa_std, usgs_std], ignore_index=True),
    geometry="geometry", crs="EPSG:4326",
)

tidal_unified.to_file(
    r"C:\Users\sriva\Desktop\CUNY_MS\FloodNet_Tides\2_tidal_analysis\tidal_unified.geojson",
    driver="GeoJSON",
)

print(f"Schema: {list(tidal_unified.columns)}")
print(f"Sources: {tidal_unified['source'].value_counts().to_dict()}")
print(f"Total rows: {len(tidal_unified)}")
print(f"Datetime range: {tidal_unified['datetime'].min()} → {tidal_unified['datetime'].max()}")
display(tidal_unified.groupby("source").head(3).reset_index(drop=True))

In [32]:
tidal_unified[tidal_unified["source"] == "usgs_observed"]["tidal_phase"].value_counts()


tidal_phase
L    4232
H    4232
Name: count, dtype: int64

In [33]:
tidal_unified[tidal_unified['source'] == 'usgs_observed']['datetime'].agg(['min', 'max'])


min   2025-01-01 07:06:00+00:00
max   2026-07-02 14:18:00+00:00
Name: datetime, dtype: datetime64[ms, UTC]